# Publisher Application

This application produces Confluence content based on an information model. 

1. Identify pages from SSOT (JSON)
1. Find all pages under root
1. Create page structure
1. Stub pages to obtain confluence page id's and to save manual work before overwriting
1. Create content (page, attachments, ...)
1. Upload content

This application uses the Publisher class defined in this folder. See ./Publisher.py

## Configuration

In [ ]:
import os
configuration_file = 'private.yaml'
assert os.path.isfile(configuration_file), 'Cannot read file ' + configuration_file

## Load configuration
Be aware not to commit your credentials!

In [ ]:
import yaml
import copy

with open(configuration_file) as f:
    config = yaml.safe_load(f)

conf_conf = config['confluence']
assert conf_conf
assert len(conf_conf['apiurl']) > 0
space_key = conf_conf['space']
root_page = conf_conf['rootpage']

conf_confidential = copy.deepcopy(config)
conf_confidential['confluence']['password'] = '***'
conf_confidential

In [ ]:
# this is the parameter cell. see cell tags. used to overwrite things with test infrastructure
confluence_username = config['confluence']['username']
confluence_password = config['confluence']['password']
cooldown = config['confluence'].get('cooldown', 0.0)
disclaimer = config['disclaimer']

## Initialize logging

In [ ]:
import logging

log = logging.getLogger()
log.setLevel(logging.CRITICAL)

LOGFILE = 'target/debug.log'

handler = logging.handlers.RotatingFileHandler(
    LOGFILE, maxBytes=(1048576*5), backupCount=7
)
formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)
log.addHandler(handler)

log.info('Starting execution')

# Setup translation

Requires gettext: `conda install gettext`

Trigger scan for translatable objects:
    `xgettext --from-code utf-8 -L python -d scan.pot templates/*`
    `msgmerge --width --update

https://phrase.com/blog/posts/translate-python-gnu-gettext/

## Internationalisation (i18n)

Publishing of content in different languages is supported. The GNU `gettext` toolchain is used to translate texts in code and templates.

Translation files are located in the 'locale' directory structure.

Use `xgettext --from-code utf-8 -L python -d confluence-publisher templates/*` to scan for content and
`msgfmt -o ../locale/de/LC_MESSAGES/confluence-publisher.mo ../locale/de/LC_MESSAGES/confluence-publisher.po` to upate the compiled translation files.

Implemtation in section 'Translation'

In [ ]:
import gettext
locale_folder = config.get('locale', './locale')
gettext.bindtextdomain('confluence-publisher', locale_folder)

In [ ]:
from pathlib import Path
translations_folders = list(Path(locale_folder).rglob("LC_MESSAGES"))
translations_folders

## Translator

Translation is done using the custom translator
It uses gettext translate when there is no translation provided in the SSOT.



In [ ]:
class Translator:
    """Translate strings"""
    logger = logging.getLogger(__name__)
    
    def __init__(self, language: str):
        self.language = language
        self.translator = gettext.translation('confluence-publisher', config.get('locale', './locale'), fallback=True, languages=[language])
        self.title_format = '{title} - [{language}]'
        
    def tr(self, element) -> str:

        if isinstance(element, dict):
            """If the provied value is a field containing translations, use them"""
            text = element.get(self.language)
            if not text and len(element.values()) > 0:
                text = list(element.values())[0]
                self.logger.warning('Falling back to {} from {}'.format(text, str(element)))
            if not text:
                return ''
            return text

        if isinstance(element, str):
            translated = self.translator.gettext(element)
            return translated

        self.logger.warning('Cannot translate element "{}" of type {}'.format(element, type(element)))
        return ''
    
    def title_language(self, title: str) -> str:
        """Create unique confluence page title per translation"""
        return self.title_format.format(title = title, language = self.language)
    
    def key_lang(self, key: str) -> str:
        return key + '-' + self.language
    
    def lang(self) -> str:
        return self.language

In [ ]:
translators = { language: Translator(language) for language in config['languages'] }
translators

In [ ]:
translators['de'].title_format = '{title}'

In [ ]:
translators['de'].tr('fadsjfklj4q9u')

In [ ]:
translators['de'].tr('Synonyms')

In [ ]:
translators['de'].tr({ 'es': 'hola'} )

In [ ]:
translators['en'].tr({ 'fr': 'salut' })

In [ ]:
translators['de'].tr('Attributes')

## Load the data
The **data** is the JSON serialized information model 

In [ ]:
import json

assert os.path.isfile(config['json']), 'The datasource in config.json {} is not a file'.format(config['json'])

data = None
with open(config['json'], 'r') as source:
     data = json.load(source)

str(data)[:512]

In [ ]:
model_languages = list(data['languages'])
model_languages

In [ ]:
# missing translations?
missing_translations = set(config['languages']) - set(model_languages)
missing_translations

In [ ]:
categories = list(data)
for category in categories:
    print('{} Elements in category "{}"'.format(len(data[category]), category))

# Compile page structure (Deprecated)

## Navigator
The navigator contains the map of all pages. It allows create confluence links from one page to it's translations and to parent pages.
The internal page map contains:
{ 'key': element key without language suffix  'page': page dictionary, 'confluence_id': if published } 

In [ ]:
class Navigator:
    
    def __init__(self):
        self.page_map = {}
        self.page_uniqueness_map = {}
    
    def register_page(self, key: str, lang: str, page: dict):
        self.page_map[self.title_lang(key, lang)] = page
        self.page_uniqueness_map[page['title']] = page
    
    def title_lang(self, key: str, lang: str) -> str:
        return key + '-' + lang
    
    def page_4_language(self, key: str, lang: str) -> str:
        return self.page_map.get(self.title_lang(key, lang))
    
    def translation_title(self, key: str, lang: str) -> str:
        """Get the title of the page with 'key' for language 'lang'"""
        return self.translation_page(key, lang)['title']

    def pages(self) -> list:
        return list(self.page_map.values())
    
    def check_page_title(self, title: str) -> bool:
        return self.page_uniqueness_map.get(title)

In [ ]:
navigator = Navigator()

## Prepare destination structure
Configuration:
- content
  - Systems
    - Tables
      - Columns

Rolled out:
- Topic 'Systems'
  - System A
    - Table A1
      - Column ID - A1
      - Colunn Name - A1
      - Column Value - A1
    - Table A2
  - System B
    - Table B1
    - Table B2

In [ ]:
navigator = Navigator()

def ensure_unique_page_title(page: dict, navigator: Navigator):
    title = page['title']
    existing_page_with_same_title = navigator.check_page_title(title)
    if existing_page_with_same_title:
        print('Page for {}[{}] has same title as {}[{}]\nOld:{}\nNew:{}'.format(
            existing_page_with_same_title['key'], existing_page_with_same_title['title'], page['key'], title,
            existing_page_with_same_title['item'], page['item']))
        tokens = title.split('-')
        if len(tokens) > 1:
            tokens.insert(len(tokens) - 1, ' ' + page['key'] + ' ')
        else:
            tokens.append(' ' + page['key'] + ' ')
        page['title'] = '-'.join(tokens)
        print('Created unique title {}'.format(page['title']))


def manual_documentation(parent_page: dict, config: dict, navigator: Navigator, translator: Translator):
    """Add a page for manual documentation, if configured"""
    md = config.get('manual-documentation')
    if md:
        page = copy.copy(parent_page)
        page_key = parent_page['key'] + '-manual-documentation'
        page['key'] = page_key
        title_format = md.get('title_format', '{parent_title} - manual')
        page['name'] = title_format.format(parent_name=parent_page['name'], key=parent_page['key'], lang=translator.lang(), parent_title=parent_page['title'])
        page['title'] = translator.title_language(page['name'])
        page['config'] = md
        # register child with parent
        parent_page['manual-documentation-page-title'] = page['title']
        
        labels = set(parent_page['labels'])
        labels.update(md.get('labels', ['manual-documentation']))
        page['labels'] = labels
        page['overwrite'] = False
        page['level'] = page['level'] + 1
        page['template'] = md.get('template')
        
        ensure_unique_page_title(page, navigator)
        navigator.register_page(page_key, translator.lang(), page)
        return page
    else:
        return None


def prepare_pages(parent_page: dict, config: dict, topic: str, level: int, navigator: Navigator, translator: Translator, data: object):
    elements = data[topic]
    pages = []
    item_filter = config.get('filter')
    filtered = 0
    for element_key in list(elements):
    
        item = data[topic][element_key]
        if item_filter:
            filter_result = eval(item_filter)
            if not filter_result:
                filtered += 1
                continue

        name_translated = translator.tr(item['name']).strip()
        if len(name_translated) > 1:
            log.warning('Strange name for item {}'.format(item))

        title_safe = name_translated
        
        if topic == 'attributes':
            parent_key = item['entity']
        elif topic == 'tables':
            parent_key = item['interface-id']
        elif topic == 'columns':
            parent_key = item['table-id']
        else:
            parent_key = parent_page['key']
        
        parent = navigator.page_4_language(parent_key, translator.lang())
        parent['child-count'] = parent['child-count'] + 1
        parent_name = parent['name']
        
        item_title_rule = config.get('title_rule')
        if item_title_rule:
            title_by_rule = eval(item_title_rule)
            if title_by_rule:
                parent_title = title_by_rule
        else:
            # Default naming rule for nested elements: {child_title} - 
            if topic in ['attributes', 'tables', 'columns']:
                title_safe = '{child_title} - {parent_name}'.format(child_title=name_translated, parent_name=parent_name)
                                              
        title = title_safe
        
        labels = set(config.get('labels', []))
        page = {
            'key': element_key,
            'topic': topic,
            'name': title,
            'title': translator.title_language(title),
            'parent': parent,
            'labels': labels,
            'item': item,
            'config': config,
            'level': level,
            'translator': translator,
            'template': config.get('template'),
            'child-count': 0,
        }
        
        ensure_unique_page_title(page, navigator)
        
        navigator.register_page(element_key, translator.lang(), page)
        pages.append(page)
        
        md = manual_documentation(page, config, navigator, translator)
        if md:
            pages.append(md)

    print('Added {} pages for topic {}. Filtered out {}'.format(len(pages), topic, filtered))
    
    # Descend into children, if any ...
    child_configurations = config.get('content')
    if child_configurations:
        """Process child types"""
        for child_config_topic in child_configurations:
            assert data[child_config_topic], 'Missing data for topic "{}"'.format(child_config_topic)
            child_config = child_configurations[child_config_topic]
            subpages = prepare_pages(page, child_config, child_config_topic, level + 1, navigator, translator, data)

    return pages


def top_level_content(config: dict, navigator: Navigator, translator: Translator, data: dict) -> list:
    """Recurse configuration content structure"""
    content = config.get('content')
    pages = []
    if content:
        for topic_key in list(content):
            topic_config = content[topic_key]
            labels = set(topic_config.get('labels', []))
            labels.add('im-parent')
            labels.add('im-parent-' + topic_key)            

            name = translator.tr(topic_config.get('title'))
            category_page = {
                'topic': topic_key + '-root',
                'key': topic_key,
                'name': name,
                'title': translator.title_language(name),
                'labels': labels,
                'config': topic_config,
                'parent': None,
                'level': 0,
                'child-count': 0,
                'translator': translator,
                'template': topic_config.get('index-template', 'parent-page-index.templ.html'),
            }
            
            ensure_unique_page_title(category_page, navigator)

            navigator.register_page(topic_key, translator.lang(), category_page)
            pages.append(category_page)
            
            print('Processing category {} [{}]'.format(topic_config.get('title'), topic_key))
                        
            sub_pages = prepare_pages(category_page, topic_config, topic_key, 1, navigator, translator, data)
    else:
        log.error('Cannot find content on root level')
    return pages


for lang in config['languages']:
    print('*** Scanning for language {} ***'.format(lang))
    pages = top_level_content(config, navigator, translators[lang], data)
    
total = len(navigator.pages())
'Will produce {} * {} ~= {} pages'.format(len(config['languages']), len(pages), total)

In [ ]:
level0 = list(filter(lambda page: page['level'] == 0, navigator.pages()))
level0

# Create graphs and render the content

In [ ]:
from tqdm.autonotebook import tqdm
from tqdm.notebook import tqdm_notebook

## Helper class to simplify template rendering

This helper is available in jina2 templates with the name 'i18n'

In [ ]:
import html
import markupsafe
from functools import reduce

class ConfluenceContentUtil:
    """This util is used in jinja2 scripts to create Confluence content"""
    def __init__(self, navigator: Navigator, translator: Translator, languages: list, json_data: dict):
        self.translator = translator
        self.navigator = navigator
        self.language = translator.language
        assert len(self.language) == 2
        self.other_lang = list(languages)
        self.other_lang.remove(self.language)
        assert len(self.other_lang) + 1 == len(languages)
        self.json_data = json_data
    
    def translate(self, text):
        return self.translator.tr(text)
    
    def translate_text(self, field) -> markupsafe.Markup:
        text = html.escape(self.translator.tr(field))
        linebreaks = text.replace('\n', '<br/>\n')
        return markupsafe.Markup(linebreaks)
        
    def other_languages(self) -> list:
        return self.other_lang
    
    def soft_link(self, key: str, lang: str = None) -> markupsafe.Markup:
        """Returns a confluence link if the key is represented with a page in the same language context or the language provided"""
        if key:
            language = lang if lang else self.language
            page = self.navigator.page_4_language(key, language)
            if page:
                return markupsafe.Markup('''<ac:link><ri:page ri:content-title="{page_title}"/><ac:plain-text-link-body><![CDATA[{name}]]></ac:plain-text-link-body></ac:link>'''.format(
                    page_title=html.escape(page['title']), name=html.escape(page.get('name'))))
            else:
                if 'DOMA' in key:
                    return translator.tr(self.json_data['domains'][key]['name'])
                elif 'COLUMN' in key:
                    return translator.tr(self.json_data['columns'][key]['name'])
                else:
                    return key
        else:
            return ''
        
    def relation_self(self, entity_key: str, relation_key: str):
        """Returns the local end of the relation_key attached to enitity_key"""
        relation = self.json_data['relations'][relation_key]
        if relation['from-to']['enti'] == entity_key:
            return relation['from-to']
        else:
            return relation['to-from']

    def relation_other(self, entity_key: str, relation_key: str):
        """Returns the remote end of the relation_key"""
        relation = self.json_data['relations'][relation_key]
        if relation['from-to']['enti'] == entity_key:
            return relation['to-from']
        else:
            return relation['from-to']
        
    def column_lineage(self, column_key: str):
        """Collects columns that are mapped with the column provided via the IM"""
        column = self.json_data['columns'][column_key]
        result = []
        for attribute_key in column['attributesmapped']:
            attribute = self.json_data['attributes'][attribute_key]
            columns_mapped = attribute['columnsmapped+']
            all_columns = map(lambda entry: columns_mapped[entry], columns_mapped)
            cols = reduce(lambda e, l: e + l, list(all_columns), [])
            result.extend(cols)
        try:
            result.remove(column_key)
        except ValueError:
            # fine if it is not in the list
            pass
        return result
    
    def attribute_lineage(self, attribute_key: str):
        """Collects columns that are mapped to the provided attribute"""
        attribute = self.json_data['attributes'][attribute_key]
        columns_mapped = attribute['columnsmapped+']
        all_columns = map(lambda entry: columns_mapped[entry], columns_mapped)
        cols = reduce(lambda e, l: e + l, list(all_columns), [])
        return cols
    
    def icon(self, key: str) -> markupsafe.Markup:
        return markupsafe.Markup(
            '<img width="50px" align="right" ' +
            'src="http://res.cloudinary.com/foryouandyourcustomers/image/upload/fyayc_icon_library/svg/0099.svg" />'
        )

test = ConfluenceContentUtil(navigator, translators['de'], ['en','fr','de'], data)
test.other_languages()

In [ ]:
from jinja2 import Environment, FileSystemLoader, select_autoescape

def create_jinja2_i18n_env(lang:str) -> Environment:
    jinja_env = Environment(
        loader=FileSystemLoader('./templates'),
        autoescape=select_autoescape(['html', 'xml']),
        extensions=["jinja2.ext.i18n"])
    
    translator = translators[lang]
    jinja_env.install_gettext_translations(translator.translator, newstyle=True)    
    util = ConfluenceContentUtil(navigator, translator, config['languages'], data)
    
    jinja_env.globals.update({ 'util': util, 'disclaimer': disclaimer, 'jinja_env': jinja_env, 'i18n': util })
    return jinja_env

i18n_environments = { lang: create_jinja2_i18n_env(lang) for lang in config['languages'] }

In [ ]:
import re

destination_folder = os.path.join('.', 'output', 'pages')
print('Writing confluence content to disk: {}'.format(destination_folder))
os.makedirs(destination_folder, exist_ok=True)

def write_page_to_disk(page, content: str, destination_folder):
    target_file = os.path.join(destination_folder, page['key'] + '-' + page['translator'].lang() + '.chtml')
    with open(target_file, 'w') as out:
        out.write(content)
    page['content'] = content
    return target_file
    
def create_graphs(page: dict, destination_folder):
    if page.get('topic') == 'entity':
        print('Would render graph for: ' + page['key'])
    pass

pages = list(navigator.pages())
with tqdm_notebook(total=len(pages), dynamic_ncols=True, unit='Page') as pbar:
    for page in pages:
        element_config = page['config']
        template_file_name = page.get('template')
        if template_file_name:
            translator = page['translator']
            jinja_env = i18n_environments[translator.language]
            jinja_template = jinja_env.get_template(template_file_name)
            
            rendered = jinja_template.render(page=page, data=data, key=page.get('key'), item=page.get('item'), 
                config=element_config, update_message = 'update')
            
            content = re.sub('<!--.*?->(\n)*', '', rendered) # strip comment lines
            ondisk = write_page_to_disk(page, content, destination_folder)
            page['file'] = ondisk

        # todo generate graph here
        create_graphs(page, destination_folder)
            
        pbar.update(1)

## Upload parallel

## Confluence integration

The [Confluence API](https://github.com/atlassian-api/atlassian-python-api) is embedded as a **git submodule** in the 'lib' folder next to this notebook.

Use `git submodule update --init` to fetch all submodules after a checkout without `--recursive` option.

If the next cell fails, install confluence-api submodule from the repository root with:
`git submodule add -f https://github.com/atlassian-api/atlassian-python-api.git notebooks/contentfactory/lib/atlassian-python-api`

In [ ]:
import sys
import os

library = 'lib/atlassian-python-api'
sys.path.insert(0, os.path.abspath(library))
from atlassian import Confluence

In [ ]:
confluence = Confluence(url=config['confluence']['apiurl'], username=confluence_username, password=confluence_password, cloud=True)
root_page_id = confluence.get_page_id(space_key, config['confluence']['rootpage'])
root_page_id

In [ ]:
from requests.exceptions import HTTPError
from colorama import Fore, Style

def print_http_error_details(e: HTTPError):
    print(Fore.RED + e.response.content.decode('utf-8'))
    from pprint import pprint
    print(Fore.YELLOW + str(vars(e)))
    pprint(vars(e.response))
    result = e.response.raw
    pprint(vars(result))
    print(Style.RESET_ALL)

In [ ]:
def upload(page: dict, content: str):
    parent = page.get('parent')
    if not parent:
        parent_page_id = root_page_id
    else:
        parent_page_id = parent.get('confluence_id')
        assert parent_page_id, 'Missing parent id for page {}'.format(page)
        
    title = page['title']
    try:
        page_id = confluence.get_page_id(space_key, title)
        current_content = confluence.get_page_by_id(page_id, expand='body.storage,ancestors,version,history')
        ancestors = current_content['ancestors']
        current_parent = None
        if len(ancestors) > 0:
            current_parent = ancestors[-1]['id']
        if current_parent != parent_page_id:
            log.warning('Moving page {title} from {source} to {destination}'.format(
                title=title, source=current_parent, destination=parent_page_id))
            confluence.move_page(space_key, page_id, target_id=parent_page_id)
            
    except Exception as e:
        create_result = confluence.create_page(space_key, title=title, parent_id=parent_page_id, body=content)
        page_id = create_result['id']
    
    try:
        confluence.update_page(page_id, title, content, minor_edit=True, version_comment='test')
    except HTTPError as error:
        log.error('Cannot publish page {}\n{}'.format(page, e.response.content.decode('utf-8')))
        print_http_error_details(error)
        
    page['confluence_id'] = page_id
    
    # TODO upload attachments
    # TODO apply labels
    
    return page['key']

In [ ]:
level0_pages = list(filter(lambda page: page['level'] == 0, navigator.pages()))
result = upload(level0_pages[2], '+stub+')

In [ ]:
import multiprocessing as mp
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

summary = None

for level in range(0,5):
    level_pages = list(filter(lambda page: page['level'] == level, navigator.pages()))
    print('Level {} has {} pages'.format(level, len(level_pages)))
    with tqdm_notebook(total=len(level_pages), dynamic_ncols=True, unit='Page') as pbar:

        def publish(page):
            content_file = page.get('file')
            if content_file:
                pbar.set_description('Uploading {} from {}'.format(page['title'], content_file))
                try:
                    with open(content_file, 'r') as content:
                        result = upload(page, str(content.read()))
                except Exception as e:
                    print(e)
            else:
                pbar.set_description('Stubbing {}'.format(page['title']))
                result = upload(page, '+stub+')
            pbar.update(1)
            return result

        with ThreadPoolExecutor(max_workers=4) as executor:
            summary = executor.map(publish, level_pages)
        pass

result = list(filter(None, summary))

In [ ]:
from tqdm.autonotebook import tqdm
from tqdm.notebook import tqdm_notebook
import time
from requests.exceptions import HTTPError

with tqdm_notebook(total=len(nodes) * len(publisher.languages), dynamic_ncols=True, unit='Page') as pbar:
    
    for lang in publisher.languages:
        
        publisher.set_language(lang)
        translator = gettext.translation('confluence-publisher', config.get('locale', './locale'), fallback=True, languages=[lang])
        translator.install()
        _ = translator.gettext
        
        language_postfix = ''
        default_parent_page_id = root_page_id
        if not publisher.is_default_language():
            # Create a node for each non default language
            language_postfix = ' [{}]'.format(lang)            
            parent_title = _('Translation ' + lang)
            labels = ('translation', 'translation-' + lang)
            result = publisher.stub(lang, parent_title, root_page_id, _('<p>Translated content to {}</p>'.format(lang)), labels)
            default_parent_page_id = result['id']
            
        # process nodes in sequence to ensure dependencies are met / parent pages exist to be referenced by children
        for node in nodes:
            title = node['title'] + language_postfix
            pbar.set_description('Preparing ' + title)

            parent = node.get('parent')
            parent_page_id = default_parent_page_id
            if parent and parent.get('confluence_page_id'):
                parent_page_id = parent['confluence_page_id'][lang]
                
            labels = node['labels']
            try:
                result = publisher.stub(node['topic'], title, parent_page_id, _('<p>Translated content to {}</p>'.format(lang)), labels)
                node.get('confluence_page_id', {})[lang] = result['id']
            except HTTPError as error:
                log.warning('Failed to stub page {title}'.format(title=title))
                cp.print_http_error_details(error)
            
            pbar.update(1)

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

cooldown = 1.0

def update_page(page: dict, lang: str):
    parent = page['parent']
    title = page['title'] + language_postfix
    pbar.set_description('Preparing ' + title)
    try:
        result = publisher.stub(page['key'], title, parent['confluence_page_id'][lang], '+stub+', page['labels'])
        page['confluence_page_id'] = result['id']
    except HTTPError as error:
        log.warning('Failed to stub page {title}'.format(title=title))
        cp.print_http_error_details(error)
    pbar.update(1)
    time.sleep(cooldown)
    return { 'page': page, 'result': result }
        

with tqdm_notebook(total=len(pages) * len(publisher.languages), dynamic_ncols=True, unit='Page') as pbar:
    for lang in publisher.languages:
        
        publisher.set_language(lang)
        translator = gettext.translation('confluence-publisher', config.get('locale', './locale'), fallback=True, languages=[lang])
        translator.install()
        _ = translator.gettext
        
        language_postfix = ''

        if not publisher.is_default_language():
            # Create a node for each non default language
            language_postfix = ' [{}]'.format(lang)
        
        with ThreadPoolExecutor(max_workers=4) as executor:
            summary = executor.map(update_page, pages, lang)
    
        pass  

result = list(filter(None, summary))

In [ ]:
from tqdm.autonotebook import tqdm
from tqdm.notebook import tqdm_notebook
import time
from requests.exceptions import HTTPError 

total = len(publisher.content_map)*len(publisher.languages)

tasklist = []

with tqdm_notebook(total=total, dynamic_ncols=True, unit='Page') as pbar:
    
    for lang in publisher.languages:
        publisher.set_language(lang)
        
        topic_dict = publisher.collect_recursive(config)
        topics = list(topic_dict)
        
        translator = gettext.translation('confluence-publisher', config.get('locale', './locale'), fallback=True, languages=[lang])
        translator.install()
        _ = translator.gettext
        
        parent_it = root_page_id
        
        parent_folder = os.path.join('.', 'output', lang)
        os.makedirs(parent_folder, exist_ok=True)

        if not publisher.is_default_language():
            parent_title = _('Translation ') + lang
            parent_it = publisher.stub(lang, parent_title, root_page_id, labels=['translation', 'im-parent', 'lang-' + lang])['id']
            print('Created language root page "{}". Confluence page id = {}'.format(parent_title, parent_it))

        for topic in topics:
            print('Processing {} in language {}'.format(topic, publisher.language))
            entry_config = topic_dict[topic]

            element_title = topic
            if entry_config.get('title'):
                element_title = entry_config['title']
            
            element_title = _(element_title)
            
            if not publisher.is_default_language():
                element_title = element_title + ' ' + lang
                
            folder_page_id = publisher.stub(topic, element_title, parent_it, labels=['topic','parent',])
            print('Created group page "{}". Confluence page id = {}'.format(element_title, str(folder_page_id['id'])))

            item_filter = entry_config.get('filter')

            for item_key in data[topic]:
                publisher.set_context(topic, item_key)

                item = data[topic][item_key]

                stub_registred = publisher.content_map[item_key].get(lang)
                if stub_registred and stub_registred.get('current_confluence_content'):
                    pbar.update(1)
                    continue

                if item_filter:
                    filter_result = eval(item_filter)
                    if not filter_result:
                        log.warning('Skipping {key} filtered out by {item_filter}'.format(key=item_key, item_filter=item_filter))
                        item['filtered'] = True
                        pbar.update(1)
                        continue

                title = publisher.page_title(item_key)
                parent = folder_page_id['id']

                # hack: attribute nested under entity
                if topic == 'attributes':
                    parent = publisher.page_for_key(item['entity'])['pageid']

                # hack: table nested under system
                if topic == 'tables':
                    parent = publisher.page_for_key(item['interface-id'])['pageid']

                # hack: column nested under table
                if topic == 'columns':
                    parent = publisher.page_for_key(item['table-id'])['pageid']

                pbar.set_description('Preparing element {}:{} with title "{}"'.format(topic, item_key, title))
                retry = 10
                while retry > 0:
                    try:
                        labels = set(entry_config.get('labels', []))
                        if lang != publisher.languages[0]:
                            labels.add('lang-' + lang)
                        #create_result = publisher.stub(item_key, title, parent, labels=list(labels))
                        
                        task = {
                            'item_key': item_key,
                            'title': title,
                            'parent': parent,
                            'type': topic,
                            'config': entry_config,
                            'language': lang
                            'labels': labels
                        }
                        
                        tasklist.append(task)
                        
                        retry = -100
                    except HTTPError as error:
                        log.warning('Failed to stub page {title}'.format(title=title))
                        cp.print_http_error_details(error)
                    except Exception as e:
                        log.warning('Cannot stub element {}:{}'.format(topic, item), e)
                        time.sleep(4.5)
                    finally:
                        retry -= 1
                
                if retry == 0:
                    log.error('Failed to stub {}'.format(item_key))
                        
                pbar.update(1)


In [ ]:
tasklist[:2]

## Now generate content

In [ ]:
def generate_page(taks):
    

In [ ]:
str(publisher.content_map)[:512]

In [ ]:
from jinja2 import Environment, FileSystemLoader, select_autoescape
import re
import time
import markupsafe

update_message = os.environ.get('UPDATE_MESSAGE', 'Automated update')

jinja_env = Environment(
    loader=FileSystemLoader('./templates'),
    autoescape=select_autoescape(['html', 'xml']),
    extensions=["jinja2.ext.i18n"]
)
jinja_env.globals.update({ 'util': publisher, 'disclaimer': disclaimer, 'update_message': update_message, 'jinja_env': jinja_env })

total = len(publisher.content_map) * len(publisher.languages)
with tqdm_notebook(total=total, dynamic_ncols=True, unit='Page') as pbar:

    topic_dict = publisher.collect_recursive(config)
    
    for lang in publisher.languages:
        # i18n
        publisher.set_language(lang)
        translator = gettext.translation('confluence-publisher', config.get('locale', './locale'), fallback=True, languages=[lang])
        translator.install()
        jinja_env.install_gettext_translations(translator, newstyle=True)
    
        topics = list(topic_dict)
        for topic in topics:

            items = list(data[topic])    
            print('Creating content for class "{}" ({}) in language "{}"'.format(topic, len(items), lang))
            topic_config = topic_dict[topic]

            jinja_template = jinja_env.get_template(topic_config['template'])

            for item_key in items:
                item = data[topic][item_key]
                publisher.set_context(topic, item_key)
                
                if item.get('filtered'):
                    pbar.update(1)
                    continue
                    
                page = publisher.page_for_key(item_key)
                if page.get('upload_successfull'):
                    pbar.update(1)
                    continue
                    
                if not item.get('icon'):
                    item['icon'] = '0612'  # default icon

                try:
                    rendered = jinja_template.render(data=data, key=item_key, item=item, config=topic_config, language=lang)
                    content_xml = re.sub('<!--.*?->(\n)*', '', rendered) # strip comment lines

                    minor_edit = False
                    current = page.get('current_confluence_content')
                    if current and current == content_xml:
                        minor_edit = True
                        pbar.set_description('Minor update on element {}:{}'.format(topic, item_key))
                        pbar.update(1)
                        continue
                    else:
                        pbar.set_description('Major update on element {}:{}'.format(topic, item_key))

                    retries = 10
                    while retries > 0:
                        try:
                            result = publisher.update_page(item_key, content_xml, minor_edit=minor_edit, version_comment=update_message)
                            page['upload_successfull'] = True
                            page['confluence_update_result'] = result
                            retries = -10
                            time.sleep(cooldown)
                        except HTTPError as error:
                            log.error('Failed to update page id {id}: {title}'.format(id=page['pageid'], title=page['title']))
                            print_http_error_details(error)
                        except ConnectionError:
                            log.excpetion('Failed to publish, retry in 4s')
                            time.sleep(5.0)
                        finally:
                            retries -= 1

                        if retries == 0:
                            item['upload_failed'] = True
                            log.error('Unable to update page id {id}: {title}\n{item}'.format(id=page['pageid'], title=page['title'], item=item))

                except Exception as e:
                    log.exception('Unable to process {} {}\n{}\n{}'.format(topic, item_key, item, content_xml), e)
                finally:
                    pbar.update(1)


In [ ]:
children = confluence.get_page_child_by_type(root_page_id, limit=1000000)
len(children)

In [ ]:
with open('target/{}.json'.format(data['model']['name']), 'w') as dest:
    json.dump(data, dest)

In [ ]:
'rlb \n sa'.replace('\n', '<br/>')

## Confluence integration

The [Confluence API](https://github.com/atlassian-api/atlassian-python-api) is embedded as a **git submodule** in the 'lib' folder next to this notebook.

Use `git submodule update --init` to fetch all submodules after a checkout without `--recursive` option.

If the next cell fails, install confluence-api submodule from the repository root with:
`git submodule add -f https://github.com/atlassian-api/atlassian-python-api.git notebooks/contentfactory/lib/atlassian-python-api`

In [ ]:
import sys
import os

library = 'lib/atlassian-python-api'
sys.path.insert(0, os.path.abspath(library))
from atlassian import Confluence

In [ ]:
confluence = Confluence(url=config['confluence']['apiurl'], username=confluence_username, password=confluence_password, cloud=True)
root_page_id = confluence.get_page_id(space_key, config['confluence']['rootpage'])
root_page_id

In [ ]:
default_language = config['languages'][0]
language_page_root = confluence.get_page_id(space_key, '')

In [ ]:
import Publisher as cp
publisher = cp.Publisher(config, data, confluence, space_key, root_page_id, languages=config['languages'])
list(map(lambda e: (e, publisher.translate(data['entities'][e]['name'])), list(data['entities'])[:3]))